# batchnorm-running-stats — ex1: EMA-update running_mean and running_var in train mode

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `batchnorm-running-stats`. Running the final beacon cell reports progress against the `CNN: BatchNorm running stats` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm running stats` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-running-stats`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-running-stats"
DD_SUBTOPIC = "CNN: BatchNorm running stats"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BatchNorm running stats (EMA) — quick refresher

BatchNorm has two distinct modes set by `model.train()` vs `model.eval()`:

**Training mode** — normalize with THIS batch's statistics, then update a rolling exponential moving average (EMA):

```
batch_mean = x.mean(dim=(0, 2, 3))                       # per-channel mean
batch_var_biased   = x.var(dim=(0, 2, 3), unbiased=False)# for the normalize step (/ N)
batch_var_unbiased = x.var(dim=(0, 2, 3), unbiased=True) # for the EMA update    (/ (N-1))
x_hat = (x - batch_mean[None, :, None, None]) / sqrt(batch_var_biased[None, :, None, None] + eps)
running_mean = (1 - momentum) * running_mean + momentum * batch_mean
running_var  = (1 - momentum) * running_var  + momentum * batch_var_unbiased   # Bessel-corrected
```

(The biased estimator gives the actual batch variance; the unbiased estimator gives a better estimate of the POPULATION variance, which is what `running_var` represents.)

**Eval mode** — normalize with the FROZEN running stats (no batch lookup, no EMA update):

```
x_hat = (x - running_mean[None, :, None, None]) / sqrt(running_var[None, :, None, None] + eps)
```

**Why `momentum` is called momentum but acts like the OPPOSITE of Adam's momentum.** PyTorch's BN convention: `running = (1-m)*running + m*batch`. So `momentum=0.1` (the default) means each batch contributes 10% of the new EMA value — a *short* memory. Closer to 0 is more inertia. Counter-intuitive but it's the API.

**Why this matters for inference.** A model deployed in eval mode with bad running stats produces garbage outputs even with correct weights. The stats are part of the model's behavior — that's why they're saved in `state_dict` (as **buffers**, not parameters — see the `register-buffer` atom).

### Exercise 1 — EMA-update running_mean and running_var in train mode

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply BatchNorm's training-mode EMA update — compute per-channel batch_mean / batch_var, then blend them into running_mean / running_var via `running = (1-m)*running + m*batch`.
> Keywords: batchnorm, running-stats, ema, momentum
> ```

**KCs targeted:** `bn-running-stats-ema-update`, `bn-per-channel-batch-stats`

Implement `ex1_bn_ema_update(x, running_mean, running_var, momentum)`. Given:

- `x`: input batch of shape `(B, C, H, W)`.
- `running_mean`, `running_var`: current EMA stats of shape `(C,)`.
- `momentum`: float in `[0, 1]`. PyTorch convention: each batch contributes `momentum` to the new EMA value.

Return a tuple `(new_running_mean, new_running_var)` computed as follows:

1. **Per-channel batch stats** — reduce over the (B, H, W) axes, leave the channel axis intact:
   ```
   batch_mean         = x.mean(dim=(0, 2, 3))
   batch_var_unbiased = x.var(dim=(0, 2, 3), unbiased=True)
   ```
   **Important PyTorch quirk:** `BatchNorm2d` uses the BIASED variance (`unbiased=False`, `/N`) for the in-batch normalize step but the UNBIASED variance (`unbiased=True`, `/(N-1)`) for the value it EMA-blends into `running_var`. This drill is about the EMA update, so use `unbiased=True` — the running stat is meant as a population-variance estimate.

2. **EMA blend** — each stat is updated as:
   ```
   new_running = (1 - momentum) * running + momentum * batch
   ```

Don't update `x` itself or do any normalization — this drill is specifically about the EMA arithmetic, isolated from the normalize step.

**Edge case.** `momentum == 0` → running stats unchanged. `momentum == 1` → running stats replaced entirely by this batch.

In [ ]:
def ex1_bn_ema_update(
    x: Tensor,
    running_mean: Tensor,
    running_var: Tensor,
    momentum: float,
) -> tuple[Tensor, Tensor]:
    batch_mean = x.mean(dim=(0, 2, 3))
    # NOTE: PyTorch uses BIASED var (/ N) for the normalize step
    # but UNBIASED var (/ (N-1)) for the running_var EMA.
    batch_var_unbiased = x.var(dim=(0, 2, 3), unbiased=True)
    new_running_mean = (1 - momentum) * running_mean + momentum * batch_mean
    new_running_var  = (1 - momentum) * running_var  + momentum * batch_var_unbiased
    return new_running_mean, new_running_var


<details><summary>Solution</summary>

```python
def ex1_bn_ema_update(
    x: Tensor,
    running_mean: Tensor,
    running_var: Tensor,
    momentum: float,
) -> tuple[Tensor, Tensor]:
    batch_mean = x.mean(dim=(0, 2, 3))
    # NOTE: PyTorch uses BIASED var (/ N) for the normalize step
    # but UNBIASED var (/ (N-1)) for the running_var EMA.
    batch_var_unbiased = x.var(dim=(0, 2, 3), unbiased=True)
    new_running_mean = (1 - momentum) * running_mean + momentum * batch_mean
    new_running_var  = (1 - momentum) * running_var  + momentum * batch_var_unbiased
    return new_running_mean, new_running_var
```

**Why `unbiased=True` for the EMA (but biased for normalize).** `BatchNorm2d` is sneaky here: it uses the BIASED variance estimator (sum-sq-deviations / N) for the in-batch normalize step `x_hat = (x - mean) / sqrt(var + eps)`, but the UNBIASED estimator (/ (N-1)) for the value it blends into `running_var` via EMA. The reason: `running_var` is what gets used at inference time as an estimate of the POPULATION variance, and the unbiased estimator is the correct one for that. Passing `unbiased=False` here would silently disagree with PyTorch's reference by a `N/(N-1)` Bessel-correction factor (a few parts per thousand for typical batch×spatial sizes).

**Why we don't update in place.** Returning fresh tensors keeps the drill pure-functional and lets the test confirm the inputs weren't clobbered. Real `BatchNorm2d` updates in-place via `.copy_()` on the registered buffers — the side-effect channel.

**Why the EMA goes UNUSED at inference.** In `model.eval()` mode, BatchNorm normalizes with `running_mean` / `running_var` and does NOT call `mean()` / `var()` on the incoming batch. So inference batches don't contribute to the EMA — only training-mode forward passes do.

**Why this is its own atom (separate from `bn-affine-params`).** Two different conditional behaviors get fused into BatchNorm: the *normalize* stage (which axes, biased or not, running or batch stats) and the *affine* stage (gamma * x_hat + beta). Drilling them in isolation lets you compose any norm variant — RMSNorm differs in normalize stage; LayerNorm differs in normalize stage; the affine stage is shared.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()